In [1]:
import SDP_interaction_inference
from SDP_interaction_inference import simulation
from SDP_interaction_inference.constraints import Constraint
from SDP_interaction_inference import optimization
from SDP_interaction_inference import correlation
from SDP_interaction_inference.dataset import Dataset
import numpy as np
import matplotlib.pyplot as plt
import tqdm
import scipy
import pandas as pd
import json

In [2]:
rng = np.random.default_rng(7)

# Simulated Data

In [3]:
# settings
interaction_values = [10.0, 5.0, 2.0, 1.5, 1.0, 0.75, 0.5, 0.25, 0.1, 0.0]
cells = 1000
rate = 1

# simulate dataset with range of interaction
data = simulation.simulate_dataset_range_BD(interaction_values, cells=cells, rate=rate)

# bootstrap
data.bootstrap(d=6)

In [4]:
# extract bounds
bounds = data.moment_bounds

# remove - from keys
bounds_dict = {f'sample{i}': val for i, val in enumerate(bounds.values())}

# save bounds
scipy.io.savemat("bounds.mat", bounds_dict)

# Reference results

In [5]:
for d in [2, 3, 4]:

    print(f"\nd = {d}\n")

    # Settings
    d_bd = d
    d_sd = d

    # Independent model free
    constraints = Constraint(
        moment_bounds=True,
        moment_matrices=True,
        factorization=True
    )
    opt_MF_ind = optimization.ModelFreeOptimization(data, d_bd=d_bd, d_me=0, d_sd=d_sd, constraints=constraints, printing=False, silent=True)
    opt_MF_ind.analyse_dataset()

    # extract status
    status = [res['status'] for res in opt_MF_ind.result_dict.values()]

    # display
    for reg, stat in zip(interaction_values, status):
        print(f"k_reg = {reg} | {stat}")


d = 2



100%|██████████| 10/10 [00:00<00:00, 18.68it/s]


k_reg = 10.0 | INFEASIBLE
k_reg = 5.0 | INFEASIBLE
k_reg = 2.0 | INFEASIBLE
k_reg = 1.5 | INFEASIBLE
k_reg = 1.0 | INFEASIBLE
k_reg = 0.75 | OPTIMAL
k_reg = 0.5 | OPTIMAL
k_reg = 0.25 | OPTIMAL
k_reg = 0.1 | OPTIMAL
k_reg = 0.0 | OPTIMAL

d = 3



100%|██████████| 10/10 [00:00<00:00, 35.28it/s]


k_reg = 10.0 | INFEASIBLE
k_reg = 5.0 | INFEASIBLE
k_reg = 2.0 | INFEASIBLE
k_reg = 1.5 | INFEASIBLE
k_reg = 1.0 | INFEASIBLE
k_reg = 0.75 | OPTIMAL
k_reg = 0.5 | INFEASIBLE
k_reg = 0.25 | OPTIMAL
k_reg = 0.1 | OPTIMAL
k_reg = 0.0 | OPTIMAL

d = 4



100%|██████████| 10/10 [00:00<00:00, 21.31it/s]

k_reg = 10.0 | INFEASIBLE
k_reg = 5.0 | INFEASIBLE
k_reg = 2.0 | INFEASIBLE
k_reg = 1.5 | INFEASIBLE
k_reg = 1.0 | INFEASIBLE
k_reg = 0.75 | OPTIMAL
k_reg = 0.5 | INFEASIBLE
k_reg = 0.25 | OPTIMAL
k_reg = 0.1 | OPTIMAL
k_reg = 0.0 | OPTIMAL


# Real Data

In [6]:
def construct_dataset(mirna_sample, mrna_dataset, beta, resamples=1000):

    # size
    gene_pairs, cells = mrna_dataset.shape

    # construct paired count dataframe
    counts_df = pd.DataFrame(
        index = [f"Gene-pair-{i}" for i in range(gene_pairs)],
        columns = [f"Cell-{j}" for j in range(cells)]
    )

    # fill with pairs
    for i in range(gene_pairs):
        gene_i = mirna_sample
        gene_j = mrna_dataset.iloc[i]
        gene_pair_ij = list(zip(gene_i, gene_j))
        counts_df.iloc[i] = gene_pair_ij

    # construct dataset object
    data = Dataset()
    data.count_dataset = counts_df
    data.cells = cells
    data.gene_pairs = gene_pairs

    # settings
    data.resamples = resamples

    # set capture
    data.beta = beta

    return data

In [7]:
# read fibroblast transcript counts
data_FIB = pd.read_csv("../Real-Data-2/Data/GSE151334_FIB_counts_thresh.csv", index_col=0)

# load capture
beta = np.loadtxt("../Real-Data-2/Capture/beta_FIB.txt")

# load RNA types
biotypes_dict = json.load(open("../Real-Data-2/Biotypes/biotypes_FIB.json"))

# select indices of protein coding mRNA and non-coding miRNA
pcRNA_indices = [idx for idx, btype in enumerate(biotypes_dict.values()) if btype == "protein_coding"]
miRNA_indices = [idx for idx, btype in enumerate(biotypes_dict.values()) if btype == "miRNA"]

# separate data
data_FIB_pcRNA = data_FIB.iloc[pcRNA_indices]
data_FIB_miRNA = data_FIB.iloc[miRNA_indices]

In [8]:
# choose RNA
miRNA = "MIR199A1"
mRNA = data_FIB_pcRNA.index[0:10]

# construct dataset of miRNA paired with mRNA
dataset_SDP = construct_dataset(data_FIB_miRNA.loc[miRNA], data_FIB_pcRNA.loc[mRNA], beta)

# bootstrap
dataset_SDP.confidence = 0.95
dataset_SDP.bootstrap(d=6, tqdm_disable=False)

100%|██████████| 10/10 [00:00<00:00, 14.54it/s]


In [9]:
# compute capture scaling matrix for d = 2
B = SDP_interaction_inference.optimization_utils.compute_B(beta, S=2, U=[], d=2)

In [10]:
# extract bounds
real_bounds = dataset_SDP.moment_bounds

# remove - from keys
real_bounds_dict = {f'sample{i}': val for i, val in enumerate(real_bounds.values())}

# add B matrix
real_bounds_dict['B'] = B

# save bounds
scipy.io.savemat("real_bounds.mat", real_bounds_dict)

# Reference results

In [11]:
for d in [2]:

    print(f"\nd = {d}\n")

    # Settings
    d_bd = d
    d_sd = d

    # Independent model free
    constraints = Constraint(
        moment_bounds=True,
        moment_matrices=True,
        factorization=True
    )
    opt_MF_ind = optimization.ModelFreeOptimization(dataset_SDP, d_bd=d_bd, d_me=0, d_sd=d_sd, constraints=constraints, printing=False, silent=True)
    opt_MF_ind.analyse_dataset()

    # display
    for res in opt_MF_ind.result_dict.values():
        print(f"Status {res['status']} | Cuts {res['cuts']} | Time {res['time']}")


d = 2



100%|██████████| 10/10 [00:00<00:00, 36.71it/s]

Status OPTIMAL | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 0 | Time 0.002000093460083008
Status INFEASIBLE | Cuts 0 | Time 0.005000114440917969
Status OPTIMAL | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 0 | Time 0.0
Status INFEASIBLE | Cuts 0 | Time 0.0


# Coefficients

Compute all coefficients and indices needed in python, store in a .mat file and simply read for MATLAB script

In [12]:
import math

In [13]:
def construct_M_s_indices(s, S, d):
    '''Moment matrix variable constructor (s): returns MATLAB indices of y.'''
    if s == 0:
        D = math.floor(d / 2)
    else:
        D = math.floor((d - 1) / 2)
    powers_D = SDP_interaction_inference.utils.compute_powers(S, D)
    powers_d = SDP_interaction_inference.utils.compute_powers(S, d)
    ND = SDP_interaction_inference.utils.compute_Nd(S, D)
    M_s = np.zeros((ND, ND), dtype=np.int64)
    e_s = [1 if i == (s - 1) else 0 for i in range(S)]
    for alpha_index, alpha in enumerate(powers_D):
        for beta_index, beta in enumerate(powers_D):
            plus = SDP_interaction_inference.utils.add_powers(alpha, beta, e_s, S=S)
            plus_index = powers_d.index(plus)
            M_s[alpha_index, beta_index] = int(plus_index + 1) # + 1 for MATLAB
    return M_s

In [14]:
def construct_factorization_indices(S, d):

    factorization = []

    powers = SDP_interaction_inference.utils.compute_powers(S, d)
    for i, alpha in enumerate(powers):

        # E[X1^a1 X2^a2] = E[X1^a1] E[X2^a2]
        if (alpha[0] > 0) and (alpha[1] > 0):
            j = powers.index([alpha[0], 0])
            l = powers.index([0, alpha[1]])
            factorization.append([i + 1, j + 1, l + 1]) # + 1 for MATLAB

    factorization = np.array(factorization, dtype=np.int64)
    return factorization

In [15]:
# choose RNA
miRNA = "MIR199A1"
mRNA = data_FIB_pcRNA.index[0:10]

# construct dataset of miRNA paired with mRNA
dataset_SDP = construct_dataset(data_FIB_miRNA.loc[miRNA], data_FIB_pcRNA.loc[mRNA], beta)

In [21]:
# choose order
d_bd = 4
d_sd = 4
d = max(d_bd, d_sd)

# bootstrap
dataset_SDP.confidence = 0.95
dataset_SDP.bootstrap(d=d_bd, tqdm_disable=False)

# extract bounds
real_bounds = dataset_SDP.moment_bounds

# remove - from keys
data_dict = {f'sample{i}': val for i, val in enumerate(real_bounds.values())}

# compute capture scaling matrix for d = d_bd
data_dict['B'] = SDP_interaction_inference.optimization_utils.compute_B(beta, S=2, U=[], d=d_bd)

# compute M_s indices
data_dict['M0'] = construct_M_s_indices(0, 2, d_sd)
data_dict['M1'] = construct_M_s_indices(1, 2, d_sd)
data_dict['M2'] = construct_M_s_indices(2, 2, d_sd)

# compute factorization indices
data_dict['factorization'] = construct_factorization_indices(2, d)

# store other useful values
data_dict['Nd'] = SDP_interaction_inference.utils.compute_Nd(2, d)
data_dict['Ndbd'] = SDP_interaction_inference.utils.compute_Nd(2, d_bd)
data_dict['Ndsd'] = SDP_interaction_inference.utils.compute_Nd(2, d_sd)

# save data
scipy.io.savemat("data.mat", data_dict)

100%|██████████| 10/10 [00:00<00:00, 23.53it/s]


# Reference results

In [22]:
# Independent model free
constraints = Constraint(
    moment_bounds=True,
    moment_matrices=True,
    factorization=True
)
opt_MF_ind = optimization.ModelFreeOptimization(dataset_SDP, d_bd=d_bd, d_me=0, d_sd=d_sd, constraints=constraints, printing=False, silent=True)
opt_MF_ind.analyse_dataset()

# display
for res in opt_MF_ind.result_dict.values():
    print(f"Status {res['status']} | Cuts {res['cuts']} | Time {res['time']}")

100%|██████████| 10/10 [00:00<00:00, 16.05it/s]

Status OPTIMAL | Cuts 5 | Time 0.03599977493286133
Status OPTIMAL | Cuts 3 | Time 0.012999773025512695
Status INFEASIBLE | Cuts 0 | Time 0.0
Status INFEASIBLE | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 1 | Time 0.003000020980834961
Status OPTIMAL | Cuts 0 | Time 0.003999948501586914
Status INFEASIBLE | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 0 | Time 0.0
Status OPTIMAL | Cuts 4 | Time 0.007999897003173828
Status INFEASIBLE | Cuts 0 | Time 0.0


# (3, 3) -> (4, 4)

Significant jump in size of M0 from 3 x 3 to 6 x 6

M1, M2 remain 3 x 3 until d_sd = 5

In [28]:
construct_M_s_indices(s=0, S=2, d=3)

array([[1, 2, 3],
       [2, 4, 5],
       [3, 5, 6]])

In [29]:
construct_M_s_indices(s=0, S=2, d=4)

array([[ 1,  2,  3,  4,  5,  6],
       [ 2,  4,  5,  7,  8,  9],
       [ 3,  5,  6,  8,  9, 10],
       [ 4,  7,  8, 11, 12, 13],
       [ 5,  8,  9, 12, 13, 14],
       [ 6,  9, 10, 13, 14, 15]])

In [39]:
construct_M_s_indices(s=1, S=2, d=4)

array([[2, 4, 5],
       [4, 7, 8],
       [5, 8, 9]])

# Birth Death

In [56]:
# choose RNA
miRNA = "MIR199A1"
mRNA = data_FIB_pcRNA.index[0:10]

# construct dataset of miRNA paired with mRNA
dataset_SDP = construct_dataset(data_FIB_miRNA.loc[miRNA], data_FIB_pcRNA.loc[mRNA], beta)

In [60]:
# choose order
d_bd = 3
d_me = 3
d_sd = 3
db = 2
d = max(d_bd, d_me, d_sd)

# bootstrap
dataset_SDP.confidence = 0.95
dataset_SDP.bootstrap(d=d_bd, tqdm_disable=False)

# extract bounds
real_bounds = dataset_SDP.moment_bounds

# remove - from keys
data_dict = {f'sample{i}': val for i, val in enumerate(real_bounds.values())}

# compute capture scaling matrix for d = d_bd
data_dict['B'] = SDP_interaction_inference.optimization_utils.compute_B(beta, S=2, U=[], d=d_bd)

# compute M_s indices
data_dict['M0'] = construct_M_s_indices(0, 2, d_sd)
data_dict['M1'] = construct_M_s_indices(1, 2, d_sd)
data_dict['M2'] = construct_M_s_indices(2, 2, d_sd)

# compute moment equation coefficients
# dummy object to access Birth-Death reaction information
dummy_opt = optimization.BirthDeathOptimization(None, d_bd=d_bd, d_me=d_me, d_sd=d_sd)
moment_powers = SDP_interaction_inference.utils.compute_powers(S=2, d=d_me - db + 1)
for i, alpha in enumerate(moment_powers):
    data_dict[f'A{i}'] = SDP_interaction_inference.optimization_utils.compute_A(alpha, dummy_opt.reactions, dummy_opt.vrs, dummy_opt.db, dummy_opt.R, dummy_opt.S, dummy_opt.d)

# store other useful values
data_dict['Nd'] = SDP_interaction_inference.utils.compute_Nd(2, d)
data_dict['Ndbd'] = SDP_interaction_inference.utils.compute_Nd(2, d_bd)
data_dict['Nda'] = SDP_interaction_inference.utils.compute_Nd(2, d_me - db + 1)
data_dict['Ndsd'] = SDP_interaction_inference.utils.compute_Nd(2, d_sd)

# save data
scipy.io.savemat("data_BD.mat", data_dict)

100%|██████████| 10/10 [00:00<00:00, 31.23it/s]


# Reference results

In [ ]:
# Birth-Death
opt_BD = optimization.BirthDeathOptimization(dataset_SDP, d_bd=d_bd, d_me=d_me, d_sd=d_sd, printing=False, silent=True, K=100, fixed=[])
opt_BD.analyse_dataset()

# display
for res in opt_BD.result_dict.values():
    print(f"Status {res['status']} | Cuts {res['cuts']} | Time {res['time']}")

100%|██████████| 10/10 [00:01<00:00,  7.09it/s]

Status OPTIMAL | Cuts 3 | Time 0.013000011444091797
Status OPTIMAL | Cuts 5 | Time 0.018999814987182617
Status INFEASIBLE | Cuts 0 | Time 0.08500003814697266
Status INFEASIBLE | Cuts 1 | Time 0.1660001277923584
Status OPTIMAL | Cuts 5 | Time 0.02700018882751465
Status OPTIMAL | Cuts 11 | Time 0.33700013160705566
Status INFEASIBLE | Cuts 2 | Time 0.06500005722045898
Status OPTIMAL | Cuts 5 | Time 0.03400015830993652
Status OPTIMAL | Cuts 3 | Time 0.0
Status INFEASIBLE | Cuts 0 | Time 0.19999980926513672


# Telegraph

In [91]:
# choose RNA
miRNA = "MIR199A1"
mRNA = data_FIB_pcRNA.index[0:1]

# construct dataset of miRNA paired with mRNA
dataset_TE = construct_dataset(data_FIB_miRNA.loc[miRNA], data_FIB_pcRNA.loc[mRNA], beta)

In [92]:
# choose order
d_bd = 3
d_me = 3
d_sd = 3
db = 2
d = max(d_bd, d_me, d_sd)

# bootstrap
dataset_TE.confidence = 0.95
dataset_TE.bootstrap(d=d_bd, tqdm_disable=False)

100%|██████████| 1/1 [00:00<00:00, 23.22it/s]


In [94]:
# Telegraph
opt_TE = optimization.TelegraphOptimization(
    dataset_TE, d_bd=d_bd, d_me=d_me, d_sd=d_sd, printing=False, silent=True,
    K=10, fixed=[], time_limit=30, total_time_limit=30
)
opt_TE.analyse_dataset()

# display
for res in opt_TE.result_dict.values():
    print(f"Status {res['status']} | Cuts {res['cuts']} | Time {res['time']}")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:28<00:00, 28.60s/it]

Status INFEASIBLE | Cuts 17 | Time 28.382999658584595


In [95]:
# Telegraph
opt_TE = optimization.TelegraphOptimization(
    dataset_TE, d_bd=d_bd, d_me=d_me, d_sd=d_sd, printing=False, silent=True,
    K=100, fixed=[], time_limit=30, total_time_limit=30
)
opt_TE.analyse_dataset()

# display
for res in opt_TE.result_dict.values():
    print(f"Status {res['status']} | Cuts {res['cuts']} | Time {res['time']}")

100%|██████████| 1/1 [00:01<00:00,  1.68s/it]

Status OPTIMAL | Cuts 28 | Time 1.3930001258850098
